# Практична робота 6. Впровадження нейронних мереж в IoT
**Варіант 1: пристрій для розпізнавання жестів** (акселерометр, 12 ознак: середнє, стандартне відхилення, мінімум і максимум за кожною з 3 осей; класи: спокій, махання, постукування, струшування; цільовий розмір моделі < 20 КБ).

Запускайте комірки зверху вниз (меню **Runtime → Run all**). Дані повністю синтетичні, фізичний датчик не потрібен.

In [ ]:
import os, json, zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)
print("TensorFlow:", tf.__version__)
os.makedirs("out", exist_ok=True)

## Завдання 1. Набір даних із мітками
Кожен приклад — це вікно акселерометра 2 с із частотою 50 Гц (100 відліків × 3 осі). Для кожного класу задано власний базовий шаблон сигналу плюс випадковий шум, випадковий нахил датчика (зсув вектора тяжіння) і випадкові амплітуди, тому класи розрізняються, але частково перекриваються. З вікна обчислюються 12 ознак: mean, std, min, max для осей X, Y, Z.

In [ ]:
FS, DUR = 50, 2.0                 # частота дискретизації (Гц), тривалість вікна (с)
N = int(FS * DUR)                 # 100 відліків у вікні
t = np.arange(N) / FS
CLASSES = ["rest", "wave", "knock", "shake"]   # спокій, махання, постукування, струшування
PER_CLASS = 250                   # >= 200 прикладів на клас

def make_window(cls):
    g = np.array([0.0, 0.0, 1.0]) + rng.normal(0, 0.12, 3)   # тяжіння (g) з випадковим нахилом
    sig = np.tile(g, (N, 1))
    noise = rng.uniform(0.04, 0.16)                           # рівень шуму датчика
    if cls == "rest" and rng.random() < 0.25:                 # іноді випадковий легкий дотик
        p = rng.integers(5, N - 15); a = rng.uniform(0.3, 0.9); k = np.arange(12)
        sig[p:p+12, 2] += a * np.exp(-k / 3.0) * np.cos(1.8 * k)
    if cls == "wave":                                         # повільні коливання 0.8-2.2 Гц, переважно по X
        f = rng.uniform(0.8, 2.2); amp = rng.uniform(0.15, 0.75); ph = rng.uniform(0, 2*np.pi, 3)
        sig[:, 0] += amp * np.sin(2*np.pi*f*t + ph[0])
        sig[:, 1] += rng.uniform(0.2, 0.6) * amp * np.sin(2*np.pi*f*t + ph[1])
        sig[:, 2] += 0.15 * amp * np.sin(2*np.pi*f*t + ph[2])
    elif cls == "knock":                                      # 1-3 короткі загасаючі удари
        for _ in range(rng.integers(1, 4)):
            p = rng.integers(5, N - 15); a = rng.uniform(0.5, 2.2); k = np.arange(12)
            pulse = a * np.exp(-k / 3.0) * np.cos(1.8 * k)
            sig[p:p+12, 2] += pulse
            sig[p:p+12, 0] += 0.3 * pulse * rng.choice([-1, 1])
            sig[p:p+12, 1] += 0.2 * pulse * rng.choice([-1, 1])
    elif cls == "shake":                                      # швидкі коливання 3.5-8 Гц по всіх осях
        f = rng.uniform(3.5, 8.0); amp = rng.uniform(0.2, 1.0)
        for ax in range(3):
            sig[:, ax] += amp * rng.uniform(0.5, 1.0) * np.sin(2*np.pi*f*t + rng.uniform(0, 2*np.pi))
    sig += rng.normal(0, noise, (N, 3))
    return sig

def features(w):                  # 12 ознак: mean, std, min, max по осях X, Y, Z
    return np.concatenate([w.mean(0), w.std(0), w.min(0), w.max(0)])

FEATURE_NAMES = [f"{s}_{a}" for s in ("mean", "std", "min", "max") for a in "XYZ"]

windows, X, y = [], [], []
for ci, c in enumerate(CLASSES):
    for _ in range(PER_CLASS):
        w = make_window(c)
        windows.append(w); X.append(features(w)); y.append(ci)
windows = np.array(windows); X = np.array(X, dtype=np.float32); y = np.array(y)
print("Вікна:", windows.shape, " ознаки:", X.shape)

# Розбиття 70 / 15 / 15 зі збереженням пропорцій класів
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=SEED)
print("Розмір навчального набору:", len(Xtr), " валідаційного:", len(Xva), " тестового:", len(Xte))
for name, yy in (("train", ytr), ("val", yva), ("test", yte)):
    print(f"{name:5s}:", {CLASSES[i]: int((yy == i).sum()) for i in range(4)})

# Нормалізація за статистикою навчального набору (ці 24 числа потім потраплять у прошивку)
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
Xtr_n, Xva_n, Xte_n = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd

In [ ]:
# Графіки: 3 зразки вікон на клас (сирий сигнал осі X) та вектори ознак
fig, axs = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
for ci, c in enumerate(CLASSES):
    idx = np.where(y == ci)[0][:3]
    for i in idx:
        axs[ci].plot(t, windows[i][:, 0], lw=1)
    axs[ci].set_ylabel(f"{c}\nX, g")
axs[-1].set_xlabel("Час, с")
fig.suptitle("Сирий сигнал акселерометра (вісь X), по 3 зразки на клас")
plt.tight_layout(); plt.savefig("out/fig1_raw.png", dpi=130); plt.show()

fig, axs = plt.subplots(2, 2, figsize=(10, 6), sharey=True)
for ci, c in enumerate(CLASSES):
    ax = axs.flat[ci]
    for i in np.where(y == ci)[0][:5]:
        ax.plot(X[i], marker="o", ms=3, lw=1)
    ax.set_title(c); ax.set_xticks(range(12)); ax.set_xticklabels(FEATURE_NAMES, rotation=90, fontsize=7)
fig.suptitle("Вектори ознак (12 значень), по 5 зразків на клас")
plt.tight_layout(); plt.savefig("out/fig2_features.png", dpi=130); plt.show()

## Завдання 2. Розробка та навчання нейронної мережі
Невелика мережа прямого поширення 12 → 32 → 16 → 4 (близько тисячі параметрів).

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(12,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(4, activation="softmax"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)
hist = model.fit(Xtr_n, ytr, validation_data=(Xva_n, yva), epochs=150, batch_size=32,
                 callbacks=[early], verbose=0)
print("Навчання завершено на епосі:", len(hist.history["loss"]))
print("Точність на навчальному (остання епоха): %.3f" % hist.history["accuracy"][-1])
print("Точність на валідаційному (остання епоха): %.3f" % hist.history["val_accuracy"][-1])

test_loss, test_acc = model.evaluate(Xte_n, yte, verbose=0)
print("Точність на ТЕСТОВОМУ наборі (float32 Keras): %.4f" % test_acc)

params = model.count_params()
float32_kb = params * 4 / 1024
model.save("out/gesture_model.keras")
print("Кількість параметрів:", params)
print("Розмір ваг float32: %.2f КБ (параметри x 4 байти)" % float32_kb)
print("Розмір файлу .keras: %.2f КБ" % (os.path.getsize("out/gesture_model.keras") / 1024))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(hist.history["loss"], label="навчання"); axs[0].plot(hist.history["val_loss"], label="валідація")
axs[0].set_title("Функція втрат"); axs[0].set_xlabel("Епоха"); axs[0].legend()
axs[1].plot(hist.history["accuracy"], label="навчання"); axs[1].plot(hist.history["val_accuracy"], label="валідація")
axs[1].set_title("Точність"); axs[1].set_xlabel("Епоха"); axs[1].legend()
plt.tight_layout(); plt.savefig("out/fig3_training.png", dpi=130); plt.show()

pred = np.argmax(model.predict(Xte_n, verbose=0), axis=1)
cm = confusion_matrix(yte, pred)
print("Матриця помилок (рядки - справжній клас, стовпці - прогноз):")
print(CLASSES); print(cm)
fig, ax = plt.subplots(figsize=(4.6, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_xticklabels(CLASSES); ax.set_yticks(range(4)); ax.set_yticklabels(CLASSES)
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
ax.set_xlabel("Прогноз"); ax.set_ylabel("Справжній клас"); ax.set_title("Матриця помилок (тест)")
plt.tight_layout(); plt.savefig("out/fig4_confusion.png", dpi=130); plt.show()

## Завдання 3. Перетворення та квантування моделі
Перетворення у TensorFlow Lite float32, потім квантування до int8 за репрезентативною вибіркою з навчального набору.

In [ ]:
# float32 TFLite
conv = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_f32 = conv.convert()
open("out/model_float32.tflite", "wb").write(tflite_f32)

# int8 TFLite (повне цілочисельне квантування входу та виходу)
def rep_data():
    for i in range(min(300, len(Xtr_n))):
        yield [Xtr_n[i:i+1].astype(np.float32)]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tflite_i8 = conv.convert()
open("out/model_int8.tflite", "wb").write(tflite_i8)

def eval_tflite(path, Xn, yy):
    it = tf.lite.Interpreter(model_path=path); it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    preds = []
    for x in Xn:
        x = x[np.newaxis, :].astype(np.float32)
        if inp["dtype"] == np.int8:
            s, z = inp["quantization"]; x = np.clip(np.round(x / s + z), -128, 127).astype(np.int8)
        it.set_tensor(inp["index"], x); it.invoke()
        o = it.get_tensor(out["index"])[0]
        if out["dtype"] == np.int8:
            s, z = out["quantization"]; o = (o.astype(np.float32) - z) * s
        preds.append(int(np.argmax(o)))
    return float(np.mean(np.array(preds) == yy))

size_f32 = os.path.getsize("out/model_float32.tflite")
size_i8 = os.path.getsize("out/model_int8.tflite")
acc_keras = float(test_acc)
acc_f32 = eval_tflite("out/model_float32.tflite", Xte_n, yte)
acc_i8 = eval_tflite("out/model_int8.tflite", Xte_n, yte)

print("%-22s %10s %12s" % ("Модель", "Розмір, Б", "Точність"))
print("%-22s %10s %12.4f" % ("Keras float32", "-", acc_keras))
print("%-22s %10d %12.4f" % ("TFLite float32", size_f32, acc_f32))
print("%-22s %10d %12.4f" % ("TFLite int8", size_i8, acc_i8))
print("Стиснення: у %.2f разу (%.2f КБ -> %.2f КБ)" % (size_f32 / size_i8, size_f32 / 1024, size_i8 / 1024))
print("Втрата точності після квантування: %.2f в.п." % (100 * (acc_f32 - acc_i8)))
print("Цільовий розмір варіанту 1: < 20 КБ ->", "ВИКОНАНО" if size_i8 < 20 * 1024 else "НЕ ВИКОНАНО")

## Завдання 4. Огляд графа моделі
Тут виводяться типи тензорів і операцій обох моделей. Графічний огляд виконайте в Netron (https://netron.app): відкрийте `model_float32.tflite` та `model_int8.tflite` і зробіть по скріншоту.

In [ ]:
for name in ("model_float32", "model_int8"):
    it = tf.lite.Interpreter(model_path=f"out/{name}.tflite"); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    print("=" * 60); print(name)
    print("  вхід :", i["shape"], i["dtype"].__name__, "quantization (scale, zero_point) =", i["quantization"])
    print("  вихід:", o["shape"], o["dtype"].__name__, "quantization (scale, zero_point) =", o["quantization"])
    print("  тензори:")
    for d in it.get_tensor_details():
        print("   ", d["index"], d["name"][:48].ljust(48), d["shape"], d["dtype"].__name__)
for name in ("model_float32", "model_int8"):
    print("=" * 60); print("Аналіз операцій:", name)
    try:
        tf.lite.experimental.Analyzer.analyze(model_path=f"out/{name}.tflite")
    except Exception as e:
        print("Analyzer недоступний:", e)

## Підсумок і завантаження результатів

In [ ]:
results = {
    "classes": CLASSES, "features": FEATURE_NAMES, "window_samples": N, "fs_hz": FS,
    "per_class": PER_CLASS, "train": int(len(Xtr)), "val": int(len(Xva)), "test": int(len(Xte)),
    "params": int(params), "float32_kb_weights": round(float32_kb, 2),
    "epochs": len(hist.history["loss"]),
    "acc_train": float(hist.history["accuracy"][-1]), "acc_val": float(hist.history["val_accuracy"][-1]),
    "acc_test_keras": acc_keras, "acc_test_tflite_f32": acc_f32, "acc_test_tflite_int8": acc_i8,
    "size_tflite_f32_bytes": int(size_f32), "size_tflite_int8_bytes": int(size_i8),
    "confusion_matrix": cm.tolist(), "tf_version": tf.__version__,
}
json.dump(results, open("out/results.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(results, ensure_ascii=False, indent=2))

with zipfile.ZipFile("pz6_results.zip", "w") as z:
    for f in sorted(os.listdir("out")):
        z.write(os.path.join("out", f), f)
print("Файли в архіві:", sorted(os.listdir("out")))
try:
    from google.colab import files
    files.download("pz6_results.zip")
except Exception as e:
    print("Завантажте pz6_results.zip вручну з панелі файлів ліворуч:", e)